# Visualização CatBoost

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


repo_root = find_repo_root(Path.cwd())
parquet_path = repo_root / "data" / "processed" / "dummy_input.parquet"
output_dir = repo_root / "src" / "prf_accidents_feature_importance" / "modeling" / "catboost" / "model"
training_summary_path = output_dir / "model_summary.json"
model_path = output_dir / "catboost_model.cbm"
visualization_dir = repo_root / "src" / "prf_accidents_feature_importance" / "modeling" / "catboost" / "visualizacao"
output_path = visualization_dir / "shap_summary.json"
plot_path = visualization_dir / "shap_summary.png"

print("Repositório detectado:", repo_root)
print("Resumo do treino:", training_summary_path)
print("Dataset esperado:", parquet_path)

if not training_summary_path.exists():
    raise FileNotFoundError("O resumo do treino não foi encontrado. Execute primeiro o notebook de treinamento em ../model/catboost_training.ipynb.")

if not parquet_path.exists():
    raise FileNotFoundError("O arquivo dummy_input.parquet não foi encontrado. Rode primeiro o notebook de geração de dados em ../data/processed/dummy_input.ipynb.")

# 1. Leitura do dataset

df = pd.read_parquet(parquet_path)
df.columns = [column.strip() for column in df.columns]

# 2. Carrega o contrato de features do treino
summary = json.loads(training_summary_path.read_text(encoding="utf-8"))
feature_columns = summary.get("feature_columns", [])
target_column = summary.get("target_column", "classificacao_acidente")

# 3. Ajusta tipos para o CatBoost
for column in feature_columns:
    if column in {"volume_pedagio", "icm_via"}:
        df[column] = df[column].astype(float)
    else:
        df[column] = df[column].astype(str)

X = df[feature_columns]
y = df[target_column]

# 4. Carrega modelo treinado, ou treina um temporário se ainda não existir
cat_features = [column for column in feature_columns if column not in {"volume_pedagio", "icm_via"}]

if model_path.exists():
    model = CatBoostClassifier()
    model.load_model(model_path)
    print("Modelo carregado de", model_path)
else:
    model = CatBoostClassifier(iterations=50, depth=4, learning_rate=0.1, loss_function="MultiClass", verbose=False)
    model.fit(X, y, cat_features=cat_features)
    model.save_model(model_path)
    print("Modelo treinado temporariamente e salvo em", model_path)

# 5. Gera explicabilidade simplificada em formato JSON e imagem
importance_values = model.get_feature_importance()
feature_importance = {
    feature: round(float(value), 4)
    for feature, value in zip(feature_columns, importance_values)
}
feature_importance = dict(sorted(feature_importance.items(), key=lambda item: item[1], reverse=True))

visualization_dir.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(feature_importance, indent=2, ensure_ascii=False), encoding="utf-8")

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(list(feature_importance.keys()), list(feature_importance.values()))
ax.set_title("Importância das features (CatBoost)")
ax.set_ylabel("Importância")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(plot_path, dpi=150)
plt.close(fig)

print(json.dumps({"summary_path": str(output_path), "plot_path": str(plot_path), "feature_importance": feature_importance}, indent=2, ensure_ascii=False))
display(pd.DataFrame.from_dict(feature_importance, orient="index", columns=["importancia"]).sort_values("importancia", ascending=False))


Repositório detectado: c:\Users\augus\Desktop\UFSC\PPGESE\Disciplinas\Ciencias de Dados\prf-accidents-feature-importance
Resumo do treino: c:\Users\augus\Desktop\UFSC\PPGESE\Disciplinas\Ciencias de Dados\prf-accidents-feature-importance\src\prf_accidents_feature_importance\modeling\catboost\model\model_summary.json
Dataset esperado: c:\Users\augus\Desktop\UFSC\PPGESE\Disciplinas\Ciencias de Dados\prf-accidents-feature-importance\data\processed\dummy_input.parquet
Modelo carregado de c:\Users\augus\Desktop\UFSC\PPGESE\Disciplinas\Ciencias de Dados\prf-accidents-feature-importance\src\prf_accidents_feature_importance\modeling\catboost\model\catboost_model.cbm
{
  "summary_path": "c:\\Users\\augus\\Desktop\\UFSC\\PPGESE\\Disciplinas\\Ciencias de Dados\\prf-accidents-feature-importance\\src\\prf_accidents_feature_importance\\modeling\\catboost\\visualizacao\\shap_summary.json",
  "plot_path": "c:\\Users\\augus\\Desktop\\UFSC\\PPGESE\\Disciplinas\\Ciencias de Dados\\prf-accidents-feature-im

,importancia
volume_pedagio,35.8469
reta,18.7997
inclinacao,14.4327
tipo_pista,13.9101
icm_via,8.2859
horario,6.5110
dia_semana,2.2138
condicao_metereologica,0.0000
